# Comparación entre redes: ¿cuál es más frágil y por qué?

Contrasta la red vial arterial (Gvial) con la red troncal de TransMilenio en espacio L y en espacio P, con métricas relativas que no dependen del tamaño de la red. Las cifras costosas se leen de los CSV que producen `01_gvial`, `02_gtransporte` y `03_robustez_gvial`; la estructura de Gvial se recalcula aquí porque toma segundos. Cada tabla trae la columna `fuente` con el archivo de origen.

## 1. Entorno

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "osmnx==2.1.1", "networkx==3.7"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/proyecto-redes-bogota")
    RUTAS = {"src": BASE / "src", "grafos": BASE / "data" / "processed", "res": BASE / "resultados",
             "referencia": BASE / "resultados" / "referencia_avance.csv"}
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUTAS = {"src": RAIZ / "src", "grafos": RAIZ / "data" / "drive_snapshot" / "processed",
             "res": RAIZ / "data" / "salidas_locales" / "resultados",
             "referencia": RAIZ / "data" / "linea_base" / "salida" / "resultados" / "referencia_avance.csv"}
GV, GT, OUT = RUTAS["res"] / "gvial", RUTAS["res"] / "gtransporte", RUTAS["res"] / "comparacion"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(RUTAS["src"]))

import pandas as pd

import adaptador_gvial as ag
import pipeline_redes as pr

UMBRAL_FC = 0.5

## 2. Estructura y concentración del flujo

La concentración usa intermediación exacta en las tres redes, para compararlas con el mismo método.

In [ ]:
_, G_v = ag.cargar_gvial(RUTAS["grafos"], "gvial_arterial", "Gvial arterial")
est_v = pr.estructura(G_v)
tramos_v = pd.read_csv(GV / "tramos_arterial.csv")
conc_v = pr.concentracion(tramos_v["eb"])
est_t = pd.read_csv(GT / "estructura_concentracion.csv", index_col=0)

filas = ["N", "M", "grado_medio", "clustering", "tiempo_medio_min", "gini", "top1"]
estructura = pd.DataFrame({
    "Gvial arterial": {**est_v, **conc_v},
    "TransMilenio L": est_t["L"].to_dict(),
    "TransMilenio P": est_t["P"].to_dict(),
}).loc[filas]
estructura["fuente"] = "Gvial: grafo del snapshot y tramos_arterial.csv | TM: estructura_concentracion.csv"
estructura.to_csv(OUT / "estructura.csv")
estructura.drop(columns="fuente").astype(float).round(3)

## 3. Fragilidad ante fallas

f_c es la fracción efectiva de tramos retirados en la que la componente gigante cae a la mitad (pasos de 1% en las tres redes). El cociente azar / adaptativo mide cuánto más daña un ataque dirigido que una falla aleatoria.

In [ ]:
def resumen_curvas(curvas: pd.DataFrame) -> dict:
    azar = curvas[curvas["estrategia"] == "aleatorio"].groupby("f")[["S", "f_efectiva", "E_rel"]].mean()
    ad = curvas[curvas["estrategia"] == "eb_recalculada"].set_index("f")
    fc_ad = pr.f_critica(ad["f_efectiva"], ad["S"], UMBRAL_FC)
    fc_az = pr.f_critica(azar["f_efectiva"], azar["S"], UMBRAL_FC)
    cinco = lambda t: t.loc[t.index.round(6) == 0.05, "E_rel"].iloc[0] - 1
    return {"f_c adaptativo": fc_ad, "f_c azar": fc_az, "azar / adaptativo": fc_az / fc_ad,
            "E con 5% retirado, adaptativo": cinco(ad), "E con 5% retirado, azar": cinco(azar)}

curvas_t = pd.read_csv(GT / "curvas_resiliencia.csv")
R_t = pd.read_csv(GT / "robustez_R.csv", index_col=0).mean()
R_v = pd.read_csv(GV / "escenarios_final.csv").set_index("escenario").loc["base", "R_media"]

fragilidad = pd.DataFrame({
    "Gvial arterial": {**resumen_curvas(pd.read_csv(GV / "curvas_resiliencia.csv")), "R": R_v},
    "TransMilenio L": {**resumen_curvas(curvas_t[curvas_t["espacio"] == "L"]), "R": R_t["L"]},
    "TransMilenio P": {**resumen_curvas(curvas_t[curvas_t["espacio"] == "P"]), "R": R_t["P"]},
})
fragilidad["fuente"] = "curvas_resiliencia.csv, escenarios_final.csv (Gvial) y robustez_R.csv (TM)"
fragilidad.to_csv(OUT / "fragilidad.csv")
fragilidad.drop(columns="fuente").astype(float).round(3)

## 4. Corredor crítico

In [ ]:
corr_v = pd.read_csv(GV / "corredores_arterial_informe.csv", index_col=0)
corr_t = pd.read_csv(GT / "troncales_L.csv", index_col=0)

def critico(corr: pd.DataFrame, col: str) -> dict:
    orden = corr[col].sort_values(ascending=False)
    return {"corredor": orden.index[0], "carga": orden.iloc[0], "segundo": orden.index[1],
            "carga primero / segundo": orden.iloc[0] / orden.iloc[1], "tipo": corr.loc[orden.index[0], "tipo"]}

criticos = pd.DataFrame({"Gvial arterial": critico(corr_v, "carga_t"), "TransMilenio L": critico(corr_t, "carga_t")})
criticos.loc["fuente"] = ["corredores_arterial_informe.csv", "troncales_L.csv"]
criticos.to_csv(OUT / "corredor_critico.csv")
criticos

## 5. Intervenciones

En Gvial, A es el enlace nuevo de 4,2 km y B la Carrera 68 y la Carrera 50 a 50 km/h; en TransMilenio, la Línea 1 del metro. El cambio de R es la media de las diferencias pareadas por réplica.

In [ ]:
ref = pd.read_csv(RUTAS["referencia"], dtype=str, keep_default_na=False).set_index("metrica")["valor_refactor"]
ruta_10 = GV / "robustez_intervenciones.csv"
if ruta_10.exists():
    rob_v = pd.read_csv(ruta_10, index_col=0)
    replicas_v = "10 réplicas (robustez_intervenciones.csv)"
else:
    final = pd.read_csv(GV / "escenarios_final.csv").set_index("escenario")
    rob_v = pd.DataFrame({"dR_medio_%": final["dR_pareado_%"], "concluyente": None})
    replicas_v = "3 réplicas (escenarios_final.csv): faltan las 10 de 03_robustez_gvial"
final_v = pd.read_csv(GV / "escenarios_final.csv").set_index("escenario")
metro = pd.read_csv(GT / "metro_l1.csv", index_col=0)["cambio_%"]

intervenciones = pd.DataFrame({
    "Gvial A (enlace 4,2 km)": {"cambio de R %": rob_v.loc["A", "dR_medio_%"], "concluyente": rob_v.loc["A", "concluyente"],
                                "cambio de eficiencia %": final_v.loc["A", "dE_%"],
                                "cambio de carga del corredor crítico %": float(ref["Carga de la NQS con A (%)"])},
    "Gvial B (Cra. 68 y 50 a 50 km/h)": {"cambio de R %": rob_v.loc["B", "dR_medio_%"], "concluyente": rob_v.loc["B", "concluyente"],
                                         "cambio de eficiencia %": final_v.loc["B", "dE_%"],
                                         "cambio de carga del corredor crítico %": float(ref["Carga de la NQS con B (%)"])},
    "TransMilenio + Metro L1": {"cambio de R %": metro["R (media de 10)"], "concluyente": "determinista",
                                "cambio de eficiencia %": metro["eficiencia entre estaciones TM"],
                                "cambio de carga del corredor crítico %": metro["carga Troncal 7"]},
})
intervenciones.loc["fuente"] = [replicas_v, replicas_v, "metro_l1.csv"]
intervenciones.to_csv(OUT / "intervenciones.csv")
intervenciones

## 6. Espacio L frente a espacio P

Las mismas métricas en los dos modelos de TransMilenio muestran qué conclusiones dependen de cómo se representa la red.

In [ ]:
L_vs_P = pd.concat([estructura[["TransMilenio L", "TransMilenio P"]].astype(float),
                    fragilidad[["TransMilenio L", "TransMilenio P"]].astype(float)])
L_vs_P["P / L"] = L_vs_P["TransMilenio P"] / L_vs_P["TransMilenio L"]
L_vs_P.to_csv(OUT / "L_vs_P.csv")
L_vs_P.round(3)

## 7. Interpretación

**Gvial frente a TransMilenio.** La red troncal de TransMilenio, vista como infraestructura (espacio L), es más frágil que la red vial arterial. Ante fallas aleatorias las dos resisten de forma parecida: la componente gigante cae a la mitad con 14,6% de los tramos retirados en Gvial y con 15,4% en TransMilenio. Ante un ataque dirigido a los tramos de mayor intermediación, TransMilenio se fragmenta con 4,1% de los tramos y Gvial con 6,2%; con 5% retirado, TransMilenio pierde 49,5% de su eficiencia y Gvial 28,5%. El ataque dirigido es 3,8 veces más eficaz que una falla aleatoria en TransMilenio y 2,4 veces en Gvial. La explicación más probable es estructural: con grado medio de 2,4, la red troncal es casi un conjunto de líneas en el que cada tramo tiene pocas alternativas, mientras que la red vial concentra más el flujo (Gini de 0,68 frente a 0,48) pero ofrece rodeos.

En las dos redes el corredor crítico es la NQS: la Avenida Carrera 30 en Gvial y la troncal 7 en TransMilenio, cuyas estaciones (NQS Calle 75, Avenida Chile, 7 de Agosto, Universidad Nacional) indican que corresponde a la NQS central. Las intervenciones sobre la red vial no cambian la robustez de forma demostrable por separado: el enlace de 4,2 km la aumenta 1,9% y la aceleración de la Carrera 68 y la Carrera 50 2,4%, con intervalos de confianza que incluyen el cero; juntas la aumentan 3,8%, con un intervalo de 1,6% a 5,9%. La aceleración sí reduce 15,1% la carga de la NQS. La Línea 1 del metro aumenta la robustez de TransMilenio en los tres tiempos de transbordo probados (entre 7% y 25%) y reduce 36% la carga de la NQS central. El tiempo medio de viaje no es comparable entre redes: Gvial usa tiempos de flujo libre y TransMilenio tiempos programados.

**Espacio L frente a espacio P.** El mismo sistema parece frágil o robusto según cómo se modele. En espacio L se fragmenta con 4,1% de los tramos retirados y su índice R es 0,444. En espacio P, donde dos estaciones se conectan si una ruta las une sin transbordo, la red no se fragmenta ni con 40% de los enlaces retirados, su índice R es 0,997 y una falla aleatoria de 5% de los enlaces le resta solo 2% de eficiencia. La diferencia proviene de la densidad: cada ruta une todas sus estaciones entre sí, de modo que P tiene 21 veces más enlaces que L. Esa robustez es en parte aparente, porque los enlaces de P comparten la infraestructura de L y el cierre de un tramo físico elimina a la vez muchas conexiones directas. Por eso la fragilidad física del sistema se mide en L, mientras P describe la redundancia de la oferta de servicios. En P el flujo se concentra mucho más (Gini de 0,93; el 1% de los enlaces lleva 31,8% de la intermediación): pocas conexiones directas estructuran los viajes más cortos.